# Modelos predictores de Implied Volatility







El **objetivo** de este notebook es aplicar diversos modelos y técnicas de IA y RRNN para encontrar el mejor modelo capaz de hacer una predicción con el mínimo error de la Volatilidad Implícita de trades de opciones del IBEX 35.

In [ ]:
from tqdm import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_management.loaders import VolatilityStepLoader
from src.enums.data_enums.database_schema.volatility_db_enum import VolatilityDBEnum
from src.enums.data_enums.option_type_enum import OptionTypeEnum

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Selección de variables y Feature Engineering

In [ ]:
volatility_df = VolatilityStepLoader.read_step_databases().copy()

# Columnas escogidas para features + target + auditoría
selected_cols = [
    VolatilityDBEnum.EXEC_DATETIME,
    VolatilityDBEnum.OPTION_CONTRACT_CODE,
    VolatilityDBEnum.OPTION_TYPE,
    VolatilityDBEnum.QUANTITY,
    VolatilityDBEnum.STRIKE_PRICE,
    VolatilityDBEnum.TRADE_TYPE,
    VolatilityDBEnum.UNDERLYING_LAG_MINUTES,
    VolatilityDBEnum.UNDERLYING_PRICE,
    VolatilityDBEnum.TIME_TO_EXPIRATION,
    VolatilityDBEnum.RATE,
    VolatilityDBEnum.IMPLIED_VOLATILITY,
]

volatility_df = volatility_df[selected_cols].copy()
print("Rows loaded:", len(volatility_df))
display(volatility_df.head())

Adicionalmente, hay que crear un catálogo de ``trade_type`` porque hay que aplicarle one-hot encoding.

In [ ]:
# Catálogo de trade_type
TRADE_TYPE_CATALOG = {
    "M": "Descripción M",
    "H": "Descripción H",
    "X": "Descripción X",
    "W": "Descripción W",
    "3": "Descripción 3",
}

TRADE_TYPE_IDS = [str(k) for k in TRADE_TYPE_CATALOG.keys()]

print("Catálogo trade_type:", len(TRADE_TYPE_IDS), "IDs ->", TRADE_TYPE_IDS)

In [ ]:
### Feature engineering ###

# Definición de columnas base para features numéricas y categóricas (one-hot)
BASE_NUMERIC_FEATURE_COLS = [
    "tte_years",
    "sqrt_tte_years",
    "log_moneyness",
    "log_moneyness_sq",
    "log_moneyness_x_sqrt_tte",
    "log_forward_moneyness",
    "rate",
    "is_call",
    "exec_hour",
    "exec_weekday",
    "underlying_lag_minutes",
    "quantity_log1p",
]

# Mapeo trade_type -> columna one-hot
TRADE_TYPE_TO_FEATURE = {tid: f"trade_type_{tid}" for tid in TRADE_TYPE_IDS}
TRADE_TYPE_FEATURE_COLS = list(TRADE_TYPE_TO_FEATURE.values())

BASE_FEATURE_COLS = BASE_NUMERIC_FEATURE_COLS + TRADE_TYPE_FEATURE_COLS
TARGET_COL = VolatilityDBEnum.IMPLIED_VOLATILITY

# Columnas auxiliares para análisis posterior (no entran al modelo)
AUX_CONTEXT_COLS = [
    VolatilityDBEnum.EXEC_DATETIME,
    VolatilityDBEnum.OPTION_CONTRACT_CODE,
]

# Construcción de features por fila (útil para explicabilidad posterior)
def build_features_from_trade(tr) -> dict:
    """
    Construye el diccionario de features por trade.
    """

    tte_years = tr[VolatilityDBEnum.TIME_TO_EXPIRATION] / 365.0
    sqrt_tte_years = np.sqrt(tte_years)

    underlying_price = tr[VolatilityDBEnum.UNDERLYING_PRICE]
    strike_price = tr[VolatilityDBEnum.STRIKE_PRICE]
    log_moneyness = np.log(underlying_price / strike_price)
    log_moneyness_sq = log_moneyness ** 2
    log_moneyness_x_sqrt_tte = log_moneyness * sqrt_tte_years

    rate = tr[VolatilityDBEnum.RATE]
    forward_price = underlying_price * np.exp(rate * tte_years)
    log_forward_moneyness = np.log(forward_price / strike_price)

    is_call = float(str(tr[VolatilityDBEnum.OPTION_TYPE]).upper() == OptionTypeEnum.CALL)

    exec_dt = tr[VolatilityDBEnum.EXEC_DATETIME]
    exec_hour = float(exec_dt.hour)
    exec_weekday = float(exec_dt.weekday())

    quantity_raw = tr[VolatilityDBEnum.QUANTITY]
    quantity_log1p = np.log1p(quantity_raw)

    underlying_lag_minutes = tr[VolatilityDBEnum.UNDERLYING_LAG_MINUTES]

    features = {
        "tte_years": tte_years,
        "sqrt_tte_years": sqrt_tte_years,
        "log_moneyness": log_moneyness,
        "log_moneyness_sq": log_moneyness_sq,
        "log_moneyness_x_sqrt_tte": log_moneyness_x_sqrt_tte,
        "log_forward_moneyness": log_forward_moneyness,
        "rate": rate,
        "is_call": is_call,
        "exec_hour": exec_hour,
        "exec_weekday": exec_weekday,
        "underlying_lag_minutes": underlying_lag_minutes,
        "quantity_log1p": quantity_log1p,
    }

    trade_value = str(tr[VolatilityDBEnum.TRADE_TYPE])
    for tid, col_name in TRADE_TYPE_TO_FEATURE.items():
        features[col_name] = float(trade_value == tid)

    return features

# Construcción de features sobre el dataset
feature_df = volatility_df.copy()

# Verificaciones auxiliares
feature_df = feature_df[
    feature_df[VolatilityDBEnum.EXEC_DATETIME].notna()
    & (feature_df[VolatilityDBEnum.UNDERLYING_PRICE] > 0)
    & (feature_df[VolatilityDBEnum.STRIKE_PRICE] > 0)
    & (feature_df[VolatilityDBEnum.TIME_TO_EXPIRATION] > 0)
    & (feature_df[VolatilityDBEnum.RATE].notna())
    & (feature_df[VolatilityDBEnum.QUANTITY].fillna(0) >= 0)
].copy()

feature_df = feature_df.sort_values(VolatilityDBEnum.EXEC_DATETIME).reset_index(drop=True)
new_features = feature_df.apply(
    lambda row: pd.Series(build_features_from_trade(row)),
    axis=1
)
model_df = pd.concat([feature_df, new_features], axis=1)

# Seleccionamos columnas finales
model_df = model_df[BASE_FEATURE_COLS + [TARGET_COL] + AUX_CONTEXT_COLS].copy()

print(f"Rows after feature engineering: {len(model_df)}")

In [ ]:
print(f"Columns ({len(model_df.columns)}): {list(model_df.columns)}")

print("\nResumen de features:")
print(f"  - Numéricas ({len(BASE_NUMERIC_FEATURE_COLS)}): {BASE_NUMERIC_FEATURE_COLS}")
print(f"  - trade_type one-hot ({len(TRADE_TYPE_FEATURE_COLS)}): {TRADE_TYPE_FEATURE_COLS}")
print(f"  - Total features modelo: {len(BASE_FEATURE_COLS)}")

print("\nColumnas auxiliares NO usadas para entrenar:", AUX_CONTEXT_COLS)

print("\n" + "="*60)
print("FEATURE STATISTICS (numéricas):")
print("="*60)
display(model_df[BASE_NUMERIC_FEATURE_COLS].describe())

print("\n" + "="*60)
print("TARGET VARIABLE (Implied Volatility):")
print("="*60)
print(f"Mean: {model_df[TARGET_COL].mean():.6f}")
print(f"Std:  {model_df[TARGET_COL].std():.6f}")
print(f"Min:  {model_df[TARGET_COL].min():.6f}")
print(f"Max:  {model_df[TARGET_COL].max():.6f}")
print(f"Missing values: {model_df[TARGET_COL].isna().sum()}")
display(model_df[[TARGET_COL]].describe())

print("\n" + "="*60)
print("DATA QUALITY:")
print("="*60)
print(f"Total rows: {len(model_df)}")
print(f"Date range: {model_df[VolatilityDBEnum.EXEC_DATETIME].min()} to {model_df[VolatilityDBEnum.EXEC_DATETIME].max()}")
missing_counts = model_df[BASE_FEATURE_COLS + [TARGET_COL]].isna().sum()
if missing_counts.sum() > 0:
    print("\nMissing values in features/target:")
    print(missing_counts[missing_counts > 0])
else:
    print("No missing values in features/target")

print("\nCorrelation with target (top 8):")
correlations = model_df[BASE_FEATURE_COLS + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)
display(correlations.head(8))

## Data split temporal

Se va a realizar un split en train validation y test, pero, a su vez, como se va a trabajar con varias ramas de familias de modelos, el conjunto de train se va splitear de nuevo usando la estrategia de los CVs consecutivos con cierto lag de fechas.

In [ ]:
# Primer split: Train, validation y test por fechas (70% - 15% - 15%)
exec_dates = model_df[VolatilityDBEnum.EXEC_DATETIME].dt.date
unique_dates_total = np.array(sorted(exec_dates.unique()))
n_dates_total = len(unique_dates_total)
train_end = int(n_dates_total * 0.70)
valid_end = int(n_dates_total * 0.85)

train_dates = unique_dates_total[:train_end]
valid_dates = unique_dates_total[train_end:valid_end]
test_dates = unique_dates_total[valid_end:]

train_df = model_df[exec_dates.isin(train_dates)].copy()
valid_df = model_df[exec_dates.isin(valid_dates)].copy()
test_df = model_df[exec_dates.isin(test_dates)].copy()

splits = [
    ("Train", train_df, train_dates),
    ("Valid", valid_df, valid_dates),
    ("Test", test_df, test_dates),
]

total_rows = len(model_df)
total_dates = len(unique_dates_total)

print("=" * 95)
print("SPLIT TEMPORAL INICIAL (sobre todo el dataset): Train / Valid / Test")
print("=" * 95)
print(f"Total filas: {total_rows:,} | Total fechas: {total_dates:,}")

for name, split_df, split_dates in splits:
    rows = len(split_df)
    dates = len(split_dates)
    start_date = split_dates[0] if dates > 0 else "N/A"
    end_date = split_dates[-1] if dates > 0 else "N/A"
    print(
        f"{name:<5} | rows: {rows:>8,} ({rows / total_rows:>7.2%}) "
        f"| dates: {dates:>5,} ({dates / total_dates:>7.2%}) "
        f"| range: {start_date} -> {end_date}"
    )

# Segundo split: k-fold por fechas para validación cruzada temporal sobre train
n_folds = 5
n_blocks = n_folds + 2  # t0..t6 => 7 bloques

date_blocks = np.array_split(train_dates, n_blocks)

folds_dates_dict = []
for i in range(n_folds):
    fold_train_dates = np.concatenate(date_blocks[: i + 2]).tolist()   # t0..t(i+1)
    fold_valid_dates = date_blocks[i + 2].tolist()                     # t(i+2)

    folds_dates_dict.append(
        (f"fold-{i+1}", {"train_dates": fold_train_dates, "valid_dates": fold_valid_dates})
    )

print("\n" + "=" * 95)
print("K-FOLDS TEMPORALES (solo dentro de Train) para validación por familia de modelos")
print("=" * 95)
print(f"Train global -> filas: {len(train_df):,} | fechas: {len(train_dates):,}")

train_exec_dates = train_df[VolatilityDBEnum.EXEC_DATETIME].dt.date

for fold_name, fold_dates in folds_dates_dict:
    fold_train_dates = fold_dates["train_dates"]
    fold_valid_dates = fold_dates["valid_dates"]

    fold_train_df = train_df[train_exec_dates.isin(fold_train_dates)]
    fold_valid_df = train_df[train_exec_dates.isin(fold_valid_dates)]

    train_rows = len(fold_train_df)
    valid_rows = len(fold_valid_df)
    train_n_dates = len(fold_train_dates)
    valid_n_dates = len(fold_valid_dates)

    train_start = fold_train_dates[0]
    train_end_date = fold_train_dates[-1]
    valid_start = fold_valid_dates[0]
    valid_end_date = fold_valid_dates[-1]

    print(f"\n{fold_name.upper()}")
    print(
        f"  TRAIN | rows: {train_rows:>8,} ({train_rows / len(train_df):>7.2%} de train) "
        f"| dates: {train_n_dates:>5,} ({train_n_dates / len(train_dates):>7.2%} de train) "
        f"| range: {train_start} -> {train_end_date}"
    )
    print(
        f"  VALID | rows: {valid_rows:>8,} ({valid_rows / len(train_df):>7.2%} de train) "
        f"| dates: {valid_n_dates:>5,} ({valid_n_dates / len(train_dates):>7.2%} de train) "
        f"| range: {valid_start} -> {valid_end_date}"
    )

## Funciones de métricas y gráficas de resultados

In [ ]:
######## CONFIGURACIONES ########
# Configuración de selección del mejor modelo
MODEL_SELECTION_CONFIG = {
    "metric": "selection_score",
    "mode": "min",
    "alpha": 0.25,
    "beta": 0.75,
    "base_metric": "rmse",
}

# Orden estándar de columnas en la tabla de resultados
# selection_score va primero; las métricas con _std se intercalan automáticamente
METRIC_ORDER = [
    "selection_score",
    "train_mae",
    "train_rmse",
    "train_r2",
    "valid_mae",
    "valid_rmse",
    "valid_r2",
]

######## MÉTRICAS ########
def calculate_regression_metrics_models(
    y_train_true,
    y_train_pred,
    y_valid_true,
    y_valid_pred,
    round_digits: int = 6,
 ):
    """
    Calcula métricas de regresión para target true y target predicted.

    Retorna un diccionario con las diferentes métricas calculadas y redondeadas
    a `round_digits` decimales.
    """
    y_train_true = np.asarray(y_train_true)
    y_train_pred = np.asarray(y_train_pred)
    y_valid_true = np.asarray(y_valid_true)
    y_valid_pred = np.asarray(y_valid_pred)

    metrics = {
        "train_mae": mean_absolute_error(y_train_true, y_train_pred),
        "train_rmse": np.sqrt(mean_squared_error(y_train_true, y_train_pred)),
        "train_r2": r2_score(y_train_true, y_train_pred),
        "valid_mae": mean_absolute_error(y_valid_true, y_valid_pred),
        "valid_rmse": np.sqrt(mean_squared_error(y_valid_true, y_valid_pred)),
        "valid_r2": r2_score(y_valid_true, y_valid_pred),
    }

    return {k: float(np.round(v, round_digits)) for k, v in metrics.items()}


def calculate_metrics_across_folds(fold_metrics_list, round_digits: int = 6):
    """
    Calcula estadísticas agregadas (media y desviación estándar) de métricas a través de los folds.
    """
    metric_keys = [k for k in METRIC_ORDER if k in fold_metrics_list[0]]
    agg_metrics = {}

    for key in metric_keys:
        values = np.asarray([fm[key] for fm in fold_metrics_list], dtype=float)
        agg_metrics[key] = float(np.round(values.mean(), round_digits))
        agg_metrics[f"{key}_std"] = float(np.round(values.std(), round_digits))

    return agg_metrics


def add_selection_score(metrics_table: pd.DataFrame, round_digits: int = 6):
    """
    Añade una métrica compuesta de selección que penaliza inestabilidad y sobreajuste.
    """

    base_metric = MODEL_SELECTION_CONFIG["base_metric"]
    alpha = MODEL_SELECTION_CONFIG["alpha"]
    beta = MODEL_SELECTION_CONFIG["beta"]

    train_metric_col = f"train_{base_metric}"
    valid_metric_col = f"valid_{base_metric}"
    valid_metric_std_col = f"{valid_metric_col}_std"

    train_metric = pd.to_numeric(metrics_table[train_metric_col])
    valid_metric = pd.to_numeric(metrics_table[valid_metric_col])
    valid_metric_std = pd.to_numeric(metrics_table[valid_metric_std_col])

    overfit_gap = np.maximum(0.0, valid_metric - train_metric)

    metrics_table = metrics_table.copy()
    metrics_table["selection_score"] = np.round(
        valid_metric + alpha * valid_metric_std + beta * overfit_gap,
        round_digits,
    )

    return metrics_table


def add_model_metrics_row(
    metrics_dict,
    model_name="model",
    metrics_table=None,
    round_digits: int = 6,
):
    """
    Agrega una fila con métricas de un modelo a la tabla comparativa.
    """
    ordered_keys = []
    for k in METRIC_ORDER:
        if k in metrics_dict:
            ordered_keys.append(k)
            if f"{k}_std" in metrics_dict:
                ordered_keys.append(f"{k}_std")

    model_row = pd.DataFrame(
        [{k: float(np.round(metrics_dict[k], round_digits)) for k in ordered_keys}],
        index=[model_name]
    )
    model_row.index.name = "model"

    if metrics_table is None:
        metrics_table = model_row
    else:
        metrics_table = pd.concat([metrics_table, model_row], axis=0)

    metrics_table = add_selection_score(metrics_table, round_digits=round_digits)

    # Orden de columnas derivado directamente de METRIC_ORDER, con _std intercalado tras cada métrica
    ordered_cols = []
    for k in METRIC_ORDER:
        if k in metrics_table.columns:
            ordered_cols.append(k)
        if f"{k}_std" in metrics_table.columns:
            ordered_cols.append(f"{k}_std")
    ordered_cols += [c for c in metrics_table.columns if c not in ordered_cols]
    metrics_table = metrics_table[ordered_cols]

    selected_metric = MODEL_SELECTION_CONFIG["metric"]
    selected_mode = MODEL_SELECTION_CONFIG["mode"]

    if selected_metric in metrics_table.columns:
        metrics_table = metrics_table.sort_values(
            by=selected_metric,
            ascending=(selected_mode == "min")
        )

    return metrics_table


def select_best_model_from_metrics_table(
    metrics_table: pd.DataFrame,
    params_registry=None,
    training_registry=None,
    family_name=None,
):
    """
    Selecciona el mejor modelo desde `metrics_table` según criterio configurable.
    """

    selected_metric = MODEL_SELECTION_CONFIG["metric"]
    selected_mode = MODEL_SELECTION_CONFIG["mode"]

    metric_series = pd.to_numeric(metrics_table[selected_metric])

    best_idx = metric_series.idxmin() if selected_mode == "min" else metric_series.idxmax()
    best_model_row = metrics_table.loc[[best_idx]].copy()

    print(f"Mejor modelo según criterio: {selected_mode}('{selected_metric}') -> {best_idx}")
    display(best_model_row)

    print(params_registry[best_idx])

    best_model_family_graphics(training_registry, best_model_row, family_name)

    return best_model_row


######## GRÁFICAS ########
def best_model_family_graphics(
    training_registry,
    best_model_row,
    family_name=None,
    sample_size: int = 7000,
):
    """
    Gráficas representativas del mejor modelo a lo largo de los folds:
    evolución de métricas train/valid, gap de sobreajuste y scatter agregado real vs pred.
    """

    best_model_name = str(best_model_row.index[0])
    plot_title = family_name if family_name is not None else best_model_name
    fold_keys = sorted(
        [k for k in training_registry.keys() if k.startswith(f"{best_model_name}_fold-")],
        key=lambda x: int(x.split("fold-")[-1]),
    )

    fold_rows = []
    all_valid_true = []
    all_valid_pred = []

    for fold_key in fold_keys:
        record = training_registry[fold_key]
        fold_name = fold_key.split("_")[-1]
        fold_metrics = record["metrics"]

        fold_rows.append(
            {
                "fold": fold_name,
                "train_mae": fold_metrics["train_mae"],
                "train_rmse": fold_metrics["train_rmse"],
                "train_r2": fold_metrics["train_r2"],
                "valid_mae": fold_metrics["valid_mae"],
                "valid_rmse": fold_metrics["valid_rmse"],
                "valid_r2": fold_metrics["valid_r2"],
                "rmse_gap": fold_metrics["valid_rmse"] - fold_metrics["train_rmse"],
            }
        )

        all_valid_true.extend(record["y_valid_true"])
        all_valid_pred.extend(record["y_valid_pred"])

    fold_metrics_df = pd.DataFrame(fold_rows)
    display(fold_metrics_df)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0, 0].plot(
        fold_metrics_df["fold"], fold_metrics_df["train_rmse"], marker="o", label="train RMSE"
    )
    axes[0, 0].plot(
        fold_metrics_df["fold"], fold_metrics_df["valid_rmse"], marker="o", label="valid RMSE"
    )
    axes[0, 0].set_title(f"{plot_title} - RMSE por fold")
    axes[0, 0].set_xlabel("Fold")
    axes[0, 0].set_ylabel("RMSE")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.25)

    axes[0, 1].plot(
        fold_metrics_df["fold"], fold_metrics_df["train_mae"], marker="o", label="train MAE"
    )
    axes[0, 1].plot(
        fold_metrics_df["fold"], fold_metrics_df["valid_mae"], marker="o", label="valid MAE"
    )
    axes[0, 1].set_title(f"{plot_title} - MAE por fold")
    axes[0, 1].set_xlabel("Fold")
    axes[0, 1].set_ylabel("MAE")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.25)

    axes[1, 0].bar(fold_metrics_df["fold"], fold_metrics_df["rmse_gap"])
    axes[1, 0].axhline(0.0, color="black", linestyle="--", linewidth=1)
    axes[1, 0].set_title(f"{plot_title} - gap de sobreajuste por fold")
    axes[1, 0].set_xlabel("Fold")
    axes[1, 0].set_ylabel("RMSE gap")
    axes[1, 0].grid(alpha=0.25)

    scatter_df = pd.DataFrame(
        {
            "y_true": np.asarray(all_valid_true, dtype=float),
            "y_pred": np.asarray(all_valid_pred, dtype=float),
        }
    )
    if len(scatter_df) > sample_size:
        scatter_df = scatter_df.sample(sample_size, random_state=42)

    min_v = float(min(scatter_df["y_true"].min(), scatter_df["y_pred"].min()))
    max_v = float(max(scatter_df["y_true"].max(), scatter_df["y_pred"].max()))

    axes[1, 1].scatter(scatter_df["y_true"], scatter_df["y_pred"], s=10, alpha=0.30)
    axes[1, 1].plot([min_v, max_v], [min_v, max_v], "k--", linewidth=1)
    axes[1, 1].set_title(f"{plot_title} - valid agregado: real vs pred")
    axes[1, 1].set_xlabel("IV real")
    axes[1, 1].set_ylabel("IV pred")
    axes[1, 1].grid(alpha=0.25)

    fig.suptitle(f"{plot_title} - resumen del mejor candidato", y=1.02)
    plt.tight_layout()
    plt.show()


## Familias de modelos

En primer lugar, vamos a preparar los sets de datos para el entrenamiento y validación que hay que hacer dentro de cada fold.

In [ ]:
def prepare_training_fold_data(train_df: pd.DataFrame, fold_info):
    """
    Prepara X/y de train y valid para un fold temporal.

    Se asume siempre este formato de entrada:
    ("fold-1", {"train_dates": [...], "valid_dates": [...]})
    """
    fold_name, fold_dates = fold_info
    fold_train_dates = fold_dates["train_dates"]
    fold_valid_dates = fold_dates["valid_dates"]

    train_exec_dates = train_df[VolatilityDBEnum.EXEC_DATETIME].dt.date

    fold_train_df = train_df[train_exec_dates.isin(fold_train_dates)].copy()
    fold_valid_df = train_df[train_exec_dates.isin(fold_valid_dates)].copy()

    if len(fold_train_df) == 0 or len(fold_valid_df) == 0:
        raise ValueError(
            f"{fold_name}: fold vacio tras filtrar fechas o nulos. "
            "Revisa train_dates/valid_dates y calidad de datos."
        )

    X_train_fold = fold_train_df[BASE_FEATURE_COLS].to_numpy(dtype=float)
    y_train_fold = fold_train_df[TARGET_COL].to_numpy(dtype=float)
    X_valid_fold = fold_valid_df[BASE_FEATURE_COLS].to_numpy(dtype=float)
    y_valid_fold = fold_valid_df[TARGET_COL].to_numpy(dtype=float)

    return X_train_fold, y_train_fold, X_valid_fold, y_valid_fold


def build_model_candidates(
    fixed_params,
    search_space=None,
    model_prefix="model",
    n_iter=40,
    seed=42,
 ):
    """
    Construye candidatos de una familia a partir de parámetros fijos y un espacio de búsqueda opcional.

    Si `search_space` está vacío o es `None`, devuelve un único modelo con `fixed_params`.
    """
    if not search_space:
        return {model_prefix: fixed_params.copy()}

    rng = np.random.default_rng(seed)
    models = {}

    for i in range(n_iter):
        sampled = {}
        for param_name, values in search_space.items():
            sampled_value = rng.choice(values)
            sampled[param_name] = (
                sampled_value.item() if isinstance(sampled_value, np.generic) else sampled_value
            )

        models[f"{model_prefix}_{i+1:03d}"] = {**fixed_params, **sampled}

    return models

In [ ]:
# Precalculo de folds para no repetir filtrado por cada modelo
fold_data_cache = []
for j_fold in range(n_folds):
    fold_name = f"fold-{j_fold+1}"
    X_train_fold, y_train_fold, X_valid_fold, y_valid_fold = prepare_training_fold_data(
        train_df, folds_dates_dict[j_fold]
    )
    fold_data_cache.append((fold_name, X_train_fold, y_train_fold, X_valid_fold, y_valid_fold))

### Lineal

Se va a probar con los siguientes:
* LinearRegression
* ElasticNetCV
* HuberRegression

In [ ]:
### Linear Regression ###
LINEAR_FIXED_PARAMS = {
    "fit_intercept": True,
    "copy_X": True,
    "n_jobs": None,
    "positive": False,
}

LINEAR_SEARCH_SPACE = {}

LINEAR_REGRESSION_CONFIG = {
    "family_name": "Linear Regression",
    "models": build_model_candidates(
        fixed_params=LINEAR_FIXED_PARAMS,
        search_space=LINEAR_SEARCH_SPACE,
        model_prefix="linear_regression",
        n_iter=1,
        seed=42,
    ),
}

total_family_models = len(LINEAR_REGRESSION_CONFIG["models"])
family_models_metrics_table = None
training_registry = {}
model_params_registry = {}

for model_i_name, model_i_params in tqdm(
    LINEAR_REGRESSION_CONFIG["models"].items(),
    total=total_family_models,
    desc=f"Entrenando familia: {LINEAR_REGRESSION_CONFIG['family_name']}",
):
    model_i_folds_metrics = []
    model_params_registry[model_i_name] = model_i_params

    for fold_name, X_train_fold, y_train_fold, X_valid_fold, y_valid_fold in fold_data_cache:
        linear_model = LinearRegression(**model_i_params)
        linear_model.fit(X_train_fold, y_train_fold)

        y_train_fold_pred = linear_model.predict(X_train_fold)
        y_valid_fold_pred = linear_model.predict(X_valid_fold)

        fold_metrics = calculate_regression_metrics_models(
            y_train_fold, y_train_fold_pred, y_valid_fold, y_valid_fold_pred
        )
        model_i_folds_metrics.append((fold_name, fold_metrics))

        training_registry[f"{model_i_name}_{fold_name}"] = {
            "model": linear_model,
            "best_iteration": None,
            "best_score": np.nan,
            "metrics": fold_metrics,
            "y_valid_true": y_valid_fold.tolist(),
            "y_valid_pred": y_valid_fold_pred.tolist(),
        }

    metrics = calculate_metrics_across_folds([m for _, m in model_i_folds_metrics])
    family_models_metrics_table = add_model_metrics_row(
        metrics,
        model_name=model_i_name,
        metrics_table=family_models_metrics_table,
    )

print("\n" + "=" * 95)
print(f"Metricas de la familia de modelos: {LINEAR_REGRESSION_CONFIG['family_name']}")
display(family_models_metrics_table.head(15))
print("=" * 95)

best_model_linear = select_best_model_from_metrics_table(
    family_models_metrics_table,
    model_params_registry,
    training_registry,
    family_name=LINEAR_REGRESSION_CONFIG["family_name"],
)


Para este modelo lineal no se pueden tunear hiperparámetros, por lo que se tiene un único modelo a probar.

### Random Forest

In [ ]:
### Random Forest ###
RF_FIXED_PARAMS = {
    "criterion": "squared_error",
    "random_state": 42,
    "n_jobs": -1,
    "min_weight_fraction_leaf": 0.0,
    "verbose": 0,
    "bootstrap": True,
}

RF_SEARCH_SPACE = {
    "n_estimators": [300, 500, 800, 1200],
    "max_depth": [None, 8, 12, 16, 24, 32],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [1.0, "sqrt", "log2", 0.7, 0.5],
    "bootstrap": [True],
    "max_samples": [None, 0.6, 0.8, 0.9],
    "min_impurity_decrease": [0.0, 1e-6, 1e-5, 1e-4],
    "ccp_alpha": [0.0, 1e-6, 1e-5, 1e-4],
}

RANDOM_FOREST_CONFIG = {
    "family_name": "Random Forest",
    "models": build_model_candidates(
        fixed_params=RF_FIXED_PARAMS,
        search_space=RF_SEARCH_SPACE,
        model_prefix="rf_rs",
        n_iter=120,
        seed=42,
    ),
}

total_family_models = len(RANDOM_FOREST_CONFIG["models"])
family_models_metrics_table = None
training_registry = {}
model_params_registry = {}

for model_i_name, model_i_params in tqdm(
    RANDOM_FOREST_CONFIG["models"].items(),
    total=total_family_models,
    desc=f"Entrenando familia: {RANDOM_FOREST_CONFIG['family_name']}",
):
    model_i_folds_metrics = []
    model_params_registry[model_i_name] = model_i_params

    for fold_name, X_train_fold, y_train_fold, X_valid_fold, y_valid_fold in fold_data_cache:
        rf_model = RandomForestRegressor(**model_i_params)
        rf_model.fit(X_train_fold, y_train_fold)

        y_train_fold_pred = rf_model.predict(X_train_fold)
        y_valid_fold_pred = rf_model.predict(X_valid_fold)

        fold_metrics = calculate_regression_metrics_models(
            y_train_fold, y_train_fold_pred, y_valid_fold, y_valid_fold_pred
        )
        model_i_folds_metrics.append((fold_name, fold_metrics))

        training_registry[f"{model_i_name}_{fold_name}"] = {
            "model": rf_model,
            "best_iteration": None,
            "best_score": np.nan,
            "metrics": fold_metrics,
            "y_valid_true": y_valid_fold.tolist(),
            "y_valid_pred": y_valid_fold_pred.tolist(),
        }

    metrics = calculate_metrics_across_folds([m for _, m in model_i_folds_metrics])
    family_models_metrics_table = add_model_metrics_row(
        metrics,
        model_name=model_i_name,
        metrics_table=family_models_metrics_table,
    )

print("\n" + "=" * 95)
print(f"Metricas de la familia de modelos: {RANDOM_FOREST_CONFIG['family_name']}")
display(family_models_metrics_table.head(15))
print("=" * 95)

best_model_rf = select_best_model_from_metrics_table(
    family_models_metrics_table,
    model_params_registry,
    training_registry,
    family_name=RANDOM_FOREST_CONFIG["family_name"],
)


### XGBoost

In [ ]:
### XGBoost ###
XGB_FIXED_PARAMS = {
    "booster": "gbtree",
    "objective": "reg:pseudohubererror",
    "eval_metric": "rmse",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
    "predictor": "auto",
    "enable_categorical": False,
    "verbosity": 0,
    "sampling_method": "uniform",
    "colsample_bylevel": 1.0,
    "colsample_bynode": 1.0,
    "max_delta_step": 0.0,
    "grow_policy": "depthwise",
    "base_score": 0.20, # Ajustado a la media del target para mejorar convergencia
}

XGB_SEARCH_SPACE = {
    "n_estimators": [400, 600, 800, 1000, 1400],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [1, 3, 5, 8, 10, 15, 20],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0.0, 0.05, 0.1, 0.2, 0.3, 0.5],
    "reg_alpha": [0.0, 0.01, 0.05, 0.1, 0.2, 0.5],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0, 3.0],
    "max_bin": [128, 256, 512],
    "num_parallel_tree": [1, 2, 4],
    "early_stopping_rounds": [30, 50, 80],
}

XGBOOST_CONFIG = {
    "family_name": "XGBoost",
    "models": build_model_candidates(
        fixed_params=XGB_FIXED_PARAMS,
        search_space=XGB_SEARCH_SPACE,
        model_prefix="xgb_rs",
        n_iter=200,
        seed=42,
    ),
}

total_family_models = len(XGBOOST_CONFIG["models"])
family_models_metrics_table = None
training_registry = {}
model_params_registry = {}

for model_i_name, model_i_params in tqdm(
    XGBOOST_CONFIG["models"].items(),
    total=total_family_models,
    desc=f"Entrenando familia: {XGBOOST_CONFIG['family_name']}",
):
    model_i_folds_metrics = []
    model_params_registry[model_i_name] = model_i_params

    for fold_name, X_train_fold, y_train_fold, X_valid_fold, y_valid_fold in fold_data_cache:
        xgb_model = XGBRegressor(**model_i_params)
        xgb_model.fit(
            X_train_fold,
            y_train_fold,
            eval_set=[(X_valid_fold, y_valid_fold)],
            verbose=False,
        )

        y_train_fold_pred = xgb_model.predict(X_train_fold)
        y_valid_fold_pred = xgb_model.predict(X_valid_fold)

        fold_metrics = calculate_regression_metrics_models(
            y_train_fold, y_train_fold_pred, y_valid_fold, y_valid_fold_pred
        )
        model_i_folds_metrics.append((fold_name, fold_metrics))

        training_registry[f"{model_i_name}_{fold_name}"] = {
            "model": xgb_model,
            "best_iteration": int(getattr(xgb_model, "best_iteration", -1)),
            "best_score": float(getattr(xgb_model, "best_score", np.nan)),
            "metrics": fold_metrics,
            "y_valid_true": y_valid_fold.tolist(),
            "y_valid_pred": y_valid_fold_pred.tolist(),
        }

    metrics = calculate_metrics_across_folds([m for _, m in model_i_folds_metrics])
    family_models_metrics_table = add_model_metrics_row(
        metrics,
        model_name=model_i_name,
        metrics_table=family_models_metrics_table,
    )

print("\n" + "=" * 95)
print(f"Metricas de la familia de modelos: {XGBOOST_CONFIG['family_name']}")
display(family_models_metrics_table.head(15))
print("=" * 95)

best_model_xgb = select_best_model_from_metrics_table(
    family_models_metrics_table,
    model_params_registry,
    training_registry,
    family_name=XGBOOST_CONFIG["family_name"],
)
